# BERTimbau para sentimento PT-BR (ML Model Lab)
Treina um Transformer, gera o model card **com os números reais** e (se você decidir) publica no Hugging Face.

**Antes de começar**
1. `Ambiente de execução` → `Alterar tipo de ambiente de execução` → **GPU (T4)** → Salvar.
2. Tenha o arquivo `ML-Model-Lab.zip` (a versão mais recente) no seu computador.
3. Para publicar: na barra lateral, ícone 🔑 **Secrets** → adicione `HF_TOKEN` com o valor do seu token **Write** (começa com `hf_`) e **ligue o Acesso ao notebook**.
4. **Rode célula por célula** (Shift+Enter). Nada demorado nem público roda sozinho: o treino completo e a publicação estão comentados.
5. **Licença:** o dataset B2W é CC BY-NC-SA 4.0, então o modelo publicado é **não comercial**.

In [ ]:
import torch
print("GPU disponível:", torch.cuda.is_available(), "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "SEM GPU: troque o ambiente de execução")

## 1) Enviar o projeto
Apaga qualquer versão antiga, pede o upload do `ML-Model-Lab.zip` e confere se a correção está presente.

In [ ]:
%cd /content
!rm -rf ML-Model-Lab ML-Model-Lab.zip
from google.colab import files
files.upload()   # escolha o ML-Model-Lab.zip
!unzip -q -o ML-Model-Lab.zip
%cd /content/ML-Model-Lab
print("Correção presente? (deve imprimir 1):")
!grep -c "warmup_steps=aquecimento" model_lab/transformer_finetune.py

## 2) Instalar e mostrar as versões

In [ ]:
!pip install -q transformers datasets accelerate scikit-learn pandas matplotlib joblib pytest
import transformers, datasets, torch
print("transformers", transformers.__version__, "| datasets", datasets.__version__, "| torch", torch.__version__)

## 3) Teste do código (segundos)
Treina um modelo **minúsculo** criado na hora e confere treino, métricas, salvamento, model card e a simulação de publicação.
**Deve terminar com `passed`.** Se falhar, **pare aqui** e me mande a mensagem de erro.

In [ ]:
!python -m pytest tests/test_transformer_smoke.py tests/test_card_publish.py -q

## 4) Ensaio rápido
Treina com 20.000 exemplos e 1 época. Você deve ver uma barra de progresso e, no final, um bloco de números (JSON).
Se der erro, leia a mensagem **desta** célula (a causa fica acima do final).

In [ ]:
!python -m model_lab.transformer_finetune --max-train 20000 --epocas 1

## 5) Ver os resultados

In [ ]:
import json, os
caminho = "results/metrics_transformer.json"
if not os.path.exists(caminho):
    print("O treino NÃO terminou: o arquivo", caminho, "não existe.")
    print("Volte à célula do ensaio e leia o erro que apareceu NELA, não nesta.")
else:
    r = json.load(open(caminho))
    print(json.dumps({k: r[k] for k in ("modelo_base", "epocas", "n_treino", "tempo_total_s", "ambiente")}, indent=1))
    print("TESTE:", json.dumps(r["teste"], indent=1))
    print("IC95 F1 macro:", r["bootstrap_teste_f1_macro"])

In [ ]:
from google.colab import files
files.download("results/metrics_transformer.json")   # guarde este arquivo (e me envie, se quiser)

## 6) Treino completo (opcional, demorado)
Só se quiser. Tire o `#` da linha de baixo. Não tenho medida do tempo; o Colab gratuito pode limitar a GPU.
O resultado do ensaio rápido já pode ser publicado: o model card mostra `n_treino = 20000`, então fica honesto.

In [ ]:
# !python -m model_lab.transformer_finetune --epocas 2

## 7) Publicação no Hugging Face (**pública e difícil de desfazer**)
Passo 1: carregar o token e definir seu usuário.

In [ ]:
import os
from google.colab import userdata
try:
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    print("Token carregado (ele não é exibido).")
except Exception as e:
    print("Não achei o Secret HF_TOKEN ou o acesso ao notebook está desligado:", type(e).__name__)
USUARIO = "SEU_USUARIO"   # <- TROQUE pelo seu usuário do Hugging Face
assert USUARIO != "SEU_USUARIO", "Troque SEU_USUARIO pelo seu usuário do Hugging Face"

Passo 2: gerar o model card a partir dos resultados reais e **ler** o que foi gerado.

In [ ]:
!python -m model_lab.make_card_transformer --usuario {USUARIO}
print(open("results/MODEL_CARD_transformer.md", encoding="utf-8").read())

Passo 3: simulação. Mostra o que seria enviado, sem enviar nada.

In [ ]:
!python -m model_lab.publish_hf --repo {USUARIO}/bertimbau-sentimento-b2w --simular

Passo 4: publicar de verdade. Tire o `#` da linha de baixo só depois de ler o card e a simulação.

In [ ]:
# !python -m model_lab.publish_hf --repo {USUARIO}/bertimbau-sentimento-b2w --eu-autorizo-publicar